# Manual domain-model comparison notebook

Helper notebook for precision, recall, F0.5, F1, and F2 calculation after manual scoring. 

Decision rule used here:
- `TP`: generated element correctly matches a reference/gold element.
- `FP`: generated element is wrong, redundant, over-specific, implementation-specific, or unsupported.
- `FN`: reference/gold elements that were missed. 

## Expected workbook structure

The notebook works best with these sheet names:

- `gold_and_reference entities`
- `gold_and_reference associations`
- `entity_scoring`
- `association_scoring`

The scoring sheets should contain at least:

```text
['dataset', 'generated', 'reference/gold ', 'decision']
```

The `dataset` column may be blank below the first row of each dataset block. The notebook fills it downward automatically.

In [7]:
from pathlib import Path
import re
import ast
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Manual counts of scoring.
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def f_beta(precision, recall, beta):
    if precision == 0 and recall == 0:
        return 0.0
    beta_sq = beta * beta
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)


MANUAL_COUNTS = [
    {"dataset": "camperplus", "element_type": "entities", "TP": 13, "FP": 16, "FN": 6},
    {"dataset": "fish_chips", "element_type": "entities", "TP": 8, "FP": 22, "FN": 1},
    {"dataset": "grocery", "element_type": "entities", "TP": 5, "FP": 25, "FN": 4},
    {"dataset": "sports", "element_type": "entities", "TP": 7, "FP": 16, "FN": 7},
    {"dataset": "education", "element_type": "entities", "TP": 12, "FP": 14, "FN": 7},
    {"dataset": "collaboration", "element_type": "entities", "TP": 9, "FP": 11, "FN": 4},
    {"dataset": "cinema", "element_type": "entities", "TP": 8, "FP": 18, "FN": 10},
    {"dataset": "matching", "element_type": "entities", "TP": 6, "FP": 25, "FN": 0},
    {"dataset": "brewery", "element_type": "entities", "TP": 8, "FP": 17, "FN": 15},

    {"dataset": "camperplus", "element_type": "associations", "TP": 4, "FP": 9, "FN": 22},
    {"dataset": "fish_chips", "element_type": "associations", "TP": 5, "FP": 7, "FN": 3},
    {"dataset": "grocery", "element_type": "associations", "TP": 0, "FP": 0, "FN": 11},
    {"dataset": "sports", "element_type": "associations", "TP": 4, "FP": 8, "FN": 9},
    {"dataset": "education", "element_type": "associations", "TP": 9, "FP": 4, "FN": 10},
    {"dataset": "collaboration", "element_type": "associations", "TP": 8, "FP": 4, "FN": 11},
    {"dataset": "cinema", "element_type": "associations", "TP": 0, "FP": 1, "FN": 21},
    {"dataset": "matching", "element_type": "associations", "TP": 2, "FP": 9, "FN": 4},
    {"dataset": "brewery", "element_type": "associations", "TP": 2, "FP": 9, "FN": 23}
]

In [8]:
# -------------------------------------------------------------------
# Count TP / FP / FN from manual decisions.
# -------------------------------------------------------------------

counts = pd.DataFrame(MANUAL_COUNTS).copy()
required = {"dataset", "element_type", "TP", "FP", "FN"}
missing = required - set(counts.columns)
if missing:
    raise ValueError(f"MANUAL_COUNTS is missing columns: {missing}")
# Add helper columns for compatibility.

counts[["TP", "FP", "FN"]] = counts[["TP", "FP", "FN"]].apply(pd.to_numeric, errors="coerce")

display(counts)

,dataset,element_type,TP,FP,FN
0,camperplus,entities,13,16,6
1,fish_chips,entities,8,22,1
2,grocery,entities,5,25,4
3,sports,entities,7,16,7
4,education,entities,12,14,7
5,collaboration,entities,9,11,4
6,cinema,entities,8,18,10
7,matching,entities,6,25,0
8,brewery,entities,8,17,15
9,camperplus,associations,4,9,22


In [9]:
# -------------------------------------------------------------------
# Calculate metrics.
# -------------------------------------------------------------------

def add_metrics(counts_df):
    df = counts_df.copy()
    df["precision"] = [safe_div(tp, tp + fp) for tp, fp in zip(df["TP"], df["FP"])]
    df["recall"] = [safe_div(tp, tp + fn) for tp, fn in zip(df["TP"], df["FN"])]
    df["F0.5"] = [f_beta(p, r, 0.5) for p, r in zip(df["precision"], df["recall"])]
    df["F1"] = [f_beta(p, r, 1.0) for p, r in zip(df["precision"], df["recall"])]
    df["F2"] = [f_beta(p, r, 2.0) for p, r in zip(df["precision"], df["recall"])]
    return df

metrics = add_metrics(counts)

# Macro averages: average of dataset-level scores.
macro = (
    metrics.groupby("element_type")[["precision", "recall", "F0.5", "F1", "F2"]]
    .mean(numeric_only=True)
    .reset_index()
)
macro.insert(0, "aggregation", "macro")

# Micro averages: sum TP/FP/FN first, then calculate metrics.
micro_counts = metrics.groupby("element_type")[["TP", "FP", "FN"]].sum().reset_index()
micro = add_metrics(micro_counts)
micro.insert(0, "aggregation", "micro")

# Overall micro across entities + associations.
overall_counts = pd.DataFrame([{
    "element_type": "overall_entities_plus_associations",
    "TP": metrics["TP"].sum(),
    "FP": metrics["FP"].sum(),
    "FN": metrics["FN"].sum(),
}])
overall_micro = add_metrics(overall_counts)
overall_micro.insert(0, "aggregation", "micro")

print("Dataset-level metrics")
display(metrics.sort_values(["element_type", "dataset"]))
print("Macro averages")
display(macro)
print("Micro averages")
display(pd.concat([micro, overall_micro], ignore_index=True))

Dataset-level metrics


,dataset,element_type,TP,FP,FN,precision,recall,F0.5,F1,F2
17,brewery,associations,2,9,23,0.181818,0.080000,0.144928,0.111111,0.090090
9,camperplus,associations,4,9,22,0.307692,0.153846,0.256410,0.205128,0.170940
15,cinema,associations,0,1,21,0.000000,0.000000,0.000000,0.000000,0.000000
14,collaboration,associations,8,4,11,0.666667,0.421053,0.597015,0.516129,0.454545
13,education,associations,9,4,10,0.692308,0.473684,0.633803,0.562500,0.505618
10,fish_chips,associations,5,7,3,0.416667,0.625000,0.446429,0.500000,0.568182
11,grocery,associations,0,0,11,0.000000,0.000000,0.000000,0.000000,0.000000
16,matching,associations,2,9,4,0.181818,0.333333,0.200000,0.235294,0.285714
12,sports,associations,4,8,9,0.333333,0.307692,0.327869,0.320000,0.312500
8,brewery,entities,8,17,15,0.320000,0.347826,0.325203,0.333333,0.341880


Macro averages


,aggregation,element_type,precision,recall,F0.5,F1,F2
0,macro,associations,0.308923,0.266068,0.289606,0.272240,0.265288
1,macro,entities,0.324304,0.638312,0.352340,0.409644,0.506185


Micro averages


,aggregation,element_type,TP,FP,FN,precision,recall,F0.5,F1,F2
0,micro,associations,34,51,114,0.400000,0.229730,0.348361,0.291845,0.251108
1,micro,entities,76,164,54,0.316667,0.584615,0.348624,0.410811,0.500000
2,micro,overall_entities_plus_associations,110,215,168,0.338462,0.395683,0.348542,0.364842,0.382742
